# FCP LoRA Training v2
## Download model directly from HuggingFace Hub

In [ ]:
# Check GPU
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('VRAM:', round(vram, 1), 'GB')

In [ ]:
# Install dependencies
!pip install -q transformers peft accelerate bitsandbytes

In [ ]:
# Upload dataset
from google.colab import files
print('Upload lora_dataset.json:')
files.upload()

In [ ]:
# Check dataset
import json
import os

DATA_PATH = 'lora_dataset.json'
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)
print('Dataset:', len(dataset), 'samples')

## Download Model from HuggingFace

In [ ]:
# Option 1: YOUR model on HuggingFace
# !rm -rf /content/ruadapt_qwen3_4b
# from huggingface_hub import snapshot_download
# MODEL_REPO = "BlackCatSpb/ruadapt-qwen3-4b"
# snapshot_download(repo_id=MODEL_REPO, local_dir="/content/ruadapt_qwen3_4b")

# Option 2: RefalMachine Ruadapt (recommended)
!rm -rf /content/ruadapt_qwen3_4b
MODEL_REPO = "RefalMachine/RuadaptQwen3-4B-Instruct"
from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_REPO, local_dir="/content/ruadapt_qwen3_4b")

print('Model downloaded')
!ls /content/ruadapt_qwen3_4b/ | head -10

## Load Model

In [ ]:
MODEL_PATH = '/content/ruadapt_qwen3_4b'
OUTPUT_DIR = '/content/fcp_adapter'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Model:', MODEL_PATH)

In [ ]:
# Load tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Vocab size:', tokenizer.vocab_size)

In [ ]:
# Load model with QLoRA
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

model = prepare_model_for_kbit_training(model)
print('Model loaded:', model.config.num_hidden_layers, 'layers')

In [ ]:
# LoRA config
RANK = 8
LORA_ALPHA = 16

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'LoRA params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Training

In [ ]:
# Dataset class
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = f"### Instruction\n{item['instruction']}\n\n### Response\n{item['output']}"
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': enc['input_ids'].squeeze()
        }

train_dataset = InstructionDataset(dataset, tokenizer)
print('Dataset ready:', len(train_dataset), 'samples')

In [ ]:
# Training config
NUM_EPOCHS = 3
BATCH_SIZE = 2
LEARNING_RATE = 3e-4

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    logging_steps=20,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    optim='adamw_bnb_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    report_to='none'
)

print(f'Config: epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

In [ ]:
# Create trainer & train
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args
)

print('Starting training...')
trainer.train()
print('Training complete!')

## Save & Download

In [ ]:
# Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

config = {
    'model_path': MODEL_PATH,
    'rank': RANK,
    'lora_alpha': LORA_ALPHA,
    'trainable_params': int(trainable),
    'total_params': int(total),
    'num_samples': len(dataset),
    'epochs': NUM_EPOCHS,
    'fcp_version': 'v15'
}
with open(f'{OUTPUT_DIR}/adapter_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')
print('Config:', config)

In [ ]:
# Download
import shutil
shutil.make_archive('/content/fcp_adapter', 'zip', '/content/fcp_adapter')
files.download('/content/fcp_adapter.zip')